# 06 — Roteamento por enum

**Trilha:** [04 ferramentas](04_ferramentas_com_funcoes.ipynb) → [05 regras e retry](05_regras_e_retry.ipynb) → **06 roteamento por enum** → [07 contrato e contexto (avançado)](../03_avancado/07_contrato_e_contexto.ipynb)

**De onde viemos:** o `TipoPedido` do [03](../01_basico/03_enums_e_guardrails.ipynb) tinha dois valores: receita ou fora do
escopo. Na prática, o usuário também pergunta preço e pede substituição. Aqui o enum cresce para **intenções**, e cada intenção
ganha uma **rota**: uma função, e as ferramentas que ela pode usar.

O padrão: o modelo **classifica** (saída pequena e validada), o código **decide** (tabela de rotas).

In [1]:
import sys
from pathlib import Path

sys.path.insert(0, str(Path.cwd().parent))  # prettifier.py e logger_general.py ficam em AI_use_cases/
from prettifier import show, show_table, styled

In [2]:
import json
from enum import StrEnum
from typing import Callable

from pydantic import BaseModel, ConfigDict, Field, ValidationError

from contrato import prompt_com_schema, saida_llm

## 1. A classificação

In [3]:
class Intencao(StrEnum):
    RECEITA = "receita"
    PRECO = "preco"
    SUBSTITUICAO = "substituicao"
    FORA_DO_ESCOPO = "fora_do_escopo"


class Classificacao(BaseModel):
    model_config = ConfigDict(extra="forbid")
    intencao: Intencao
    confianca: float = Field(ge=0, le=1, description="Quão certo você está da intenção, de 0 a 1")
    prato: str | None = None
    ingrediente: str | None = None


show(f"Intencao no schema: {Classificacao.model_json_schema()['$defs']['Intencao']['enum']}")

Intencao no schema: ['receita', 'preco', 'substituicao', 'fora_do_escopo']


## 2. A tabela de rotas

Uma função por intenção. Aqui elas só descrevem o que fariam; no sistema real, `responder_receita` chamaria o `pedir_lista` do
[05](05_regras_e_retry.ipynb), e `responder_preco` o laço de ferramentas do [04](04_ferramentas_com_funcoes.ipynb).

In [4]:
def responder_receita(c: Classificacao) -> str:
    return f"gerar lista de compras para {c.prato!r}"


def responder_preco(c: Classificacao) -> str:
    return f"consultar preço de {c.ingrediente!r}"


def responder_substituicao(c: Classificacao) -> str:
    return f"sugerir substituto para {c.ingrediente!r} (verificando a despensa)"


def recusar(c: Classificacao) -> str:
    return "recusar com educação: só ajudo com compras de mercado"


ROTAS: dict[Intencao, Callable[[Classificacao], str]] = {
    Intencao.RECEITA: responder_receita,
    Intencao.PRECO: responder_preco,
    Intencao.SUBSTITUICAO: responder_substituicao,
    Intencao.FORA_DO_ESCOPO: recusar,
}

faltando = set(Intencao) - ROTAS.keys()
assert not faltando, f"intenções sem rota: {faltando}"
show(f"todas as {len(Intencao)} intenções têm rota")

todas as 4 intenções têm rota


O `assert` logo depois da tabela é um **teste de cobertura**: se alguém acrescentar uma intenção e esquecer a rota, o notebook
(ou o módulo, ao ser importado) falha na hora — não quando um usuário cair no caso novo.

## 3. Ferramentas por rota (menor privilégio)

O 04 deu todas as ferramentas ao modelo. Com rotas, cada caminho recebe **só o que precisa**:

In [5]:
class Ferramenta(StrEnum):  # o mesmo enum do 04
    CONSULTAR_PRECO = "consultar_preco"
    VERIFICAR_DESPENSA = "verificar_despensa"


FERRAMENTAS_POR_ROTA: dict[Intencao, frozenset[Ferramenta]] = {
    Intencao.RECEITA: frozenset({Ferramenta.VERIFICAR_DESPENSA}),
    Intencao.PRECO: frozenset({Ferramenta.CONSULTAR_PRECO}),
    Intencao.SUBSTITUICAO: frozenset({Ferramenta.VERIFICAR_DESPENSA}),
    Intencao.FORA_DO_ESCOPO: frozenset(),
}
assert FERRAMENTAS_POR_ROTA.keys() == set(Intencao)


def ferramenta_permitida(intencao: Intencao, nome: str) -> bool:
    return nome in FERRAMENTAS_POR_ROTA[intencao]


show_table(["intenção", "rota", "ferramentas permitidas"],
       [(i, ROTAS[i].__name__, ", ".join(sorted(f.value for f in FERRAMENTAS_POR_ROTA[i])) or "nenhuma") for i in Intencao])
show(f"ferramenta_permitida(RECEITA, 'consultar_preco'): {ferramenta_permitida(Intencao.RECEITA, 'consultar_preco')}")

                                                                    
  intenção         rota                     ferramentas permitidas  
 ────────────────────────────────────────────────────────────────── 
  receita          responder_receita        verificar_despensa      
  preco            responder_preco          consultar_preco         
  substituicao     responder_substituicao   verificar_despensa      
  fora_do_escopo   recusar                  nenhuma                 
                                                                    
ferramenta_permitida(RECEITA, 'consultar_preco'): False


Restringir a lista de ferramentas enviada **não basta**: o modelo pode pedir uma ferramenta que não recebeu. A checagem
`ferramenta_permitida` vai no momento de executar, junto do guardrail do 04.

## 4. Roteando, com limite de confiança

In [6]:
LIMIAR_CONFIANCA = 0.7


def rotear(c: Classificacao) -> str:
    if c.confianca < LIMIAR_CONFIANCA:
        return f"revisão humana (confiança {c.confianca:.2f} < {LIMIAR_CONFIANCA})"
    return ROTAS[c.intencao](c)


INSTRUCAO = ("Classifique o pedido do usuário de um assistente de compras de mercado. "
             "Preencha 'prato' quando houver um prato e 'ingrediente' quando houver um ingrediente.")

In [7]:
# api: configuração do cliente — lê a chave e o endpoint do .env
import os

from dotenv import load_dotenv
from openai import OpenAI

load_dotenv()
api_key = os.getenv("openai")
if not api_key:
    raise ValueError("variável 'openai' (chave da API) não encontrada no .env")
openai_base_url = os.getenv("openai_base_url")
if not openai_base_url:
    raise ValueError("variável 'openai_base_url' não encontrada no .env")

AI_MODEL = "gpt-4o-mini"
ai_client = OpenAI(api_key=api_key, base_url=openai_base_url)

In [ ]:
# api: classificar pedidos reais
pedidos = [
    "Pizza margherita para 4 pessoas",
    "Quanto está o quilo da muçarela?",
    "Não tenho manjericão, uso o quê?",
    "Me indica um filme",
]
for pedido in pedidos:
    texto = ai_client.chat.completions.create(
        model=AI_MODEL,
        messages=[{"role": "system", "content": prompt_com_schema(INSTRUCAO, Classificacao)},
                  {"role": "user", "content": pedido}],
    ).choices[0].message.content
    classificacao = saida_llm(Classificacao).validate_python(texto)
    show(f"{pedido!r} -> {classificacao.intencao} ({classificacao.confianca:.2f}) -> {rotear(classificacao)}")

Sem chave, com classificações **construídas** no formato do contrato:

In [7]:
construidas = {
    "Pizza margherita para 4 pessoas": '{"intencao": "receita", "confianca": 0.95, "prato": "pizza margherita"}',
    "Quanto está o quilo da muçarela?": '{"intencao": "preco", "confianca": 0.9, "ingrediente": "muçarela"}',
    "Não tenho manjericão, uso o quê?": '{"intencao": "substituicao", "confianca": 0.85, "ingrediente": "manjericão"}',
    "Me indica um filme": '{"intencao": "fora_do_escopo", "confianca": 0.97}',
    "Tem algo pra janta?": '{"intencao": "receita", "confianca": 0.4}',
}
linhas = []
for pedido, texto in construidas.items():
    c = Classificacao.model_validate_json(texto)
    tom = "good" if c.confianca >= LIMIAR_CONFIANCA else "warn"
    linhas.append((pedido, c.intencao, styled(f"{c.confianca:.2f}", tom), rotear(c)))
show_table(["pedido", "intenção", "confiança", "decisão"], linhas)

                                                                                                              
  pedido                             intenção         confiança   decisão                                     
 ──────────────────────────────────────────────────────────────────────────────────────────────────────────── 
  Pizza margherita para 4 pessoas    receita          0.95        gerar lista de compras para 'pizza          
                                                                  margherita'                                 
  Quanto está o quilo da muçarela?   preco            0.90        consultar preço de 'muçarela'               
  Não tenho manjericão, uso o quê?   substituicao     0.85        sugerir substituto para 'manjericão'        
                                                                  (verificando a despensa)                    
  Me indica um filme                 fora_do_escopo   0.97        recusar com educação: só ajudo com compras  
 

## ⚠️ O que ainda quebra

**1. A confiança é o modelo falando de si mesmo.** Não é uma probabilidade calibrada. Um erro pode vir com 0,95 e passar direto
pelo limiar (*exemplo construído*: "bomba de chocolate" é um doce):

In [8]:
errada_e_confiante = Classificacao.model_validate_json('{"intencao": "fora_do_escopo", "confianca": 0.95}')
show(f"'Como faço bomba de chocolate?' -> {rotear(errada_e_confiante)}")

'Como faço bomba de chocolate?' -> recusar com educação: só ajudo com compras de mercado


O limiar pega a dúvida **declarada**, não o erro. Para medir de verdade é preciso um conjunto de pedidos com a resposta certa
conhecida e conferir a taxa de acerto — o [10](../03_avancado/10_pipeline_completo.ipynb) monta esse tipo de teste.

**2. Rota faltando sem o teste de cobertura = `KeyError` em produção.**

In [9]:
class IntencaoV2(StrEnum):
    RECEITA = "receita"
    CARDAPIO_DA_SEMANA = "cardapio_da_semana"  # nova, sem rota


rotas_v2 = {IntencaoV2.RECEITA: lambda c: "lista"}
try:
    rotas_v2[IntencaoV2.CARDAPIO_DA_SEMANA]
except KeyError as ex:
    show(f"KeyError: {ex}")
show(f"o teste teria pego: {set(IntencaoV2) - rotas_v2.keys()}")

KeyError: <IntencaoV2.CARDAPIO_DA_SEMANA: 'cardapio_da_semana'>
o teste teria pego: {<IntencaoV2.CARDAPIO_DA_SEMANA: 'cardapio_da_semana'>}


**3. O valor do enum é um formato gravado.** Ele aparece no prompt, no JSON e em logs. Trocar `"preco"` por `"preço"` "só para
ficar bonito" quebra a leitura de tudo que já foi salvo:

In [10]:
class IntencaoRenomeada(StrEnum):
    PRECO = "preço"


log_antigo = '{"intencao": "preco"}'
try:
    IntencaoRenomeada(json.loads(log_antigo)["intencao"])
except ValueError as ex:
    show(f"ValueError: {ex}")

ValueError: 'preco' is not a valid IntencaoRenomeada


Regra prática: o **nome** do membro (`PRECO`) pode mudar à vontade; o **valor** (`"preco"`) é contrato com o mundo de fora.

**4. Valor desconhecido derruba a classificação inteira.** Se o modelo inventar `"intencao": "receita_doce"`, a validação falha
e o pedido some. Às vezes é melhor aceitar e mandar para uma rota segura — o avançado mostra o enum com `DESCONHECIDO`.

In [11]:
try:
    Classificacao.model_validate_json('{"intencao": "receita_doce", "confianca": 0.9}')
except ValidationError as ex:
    show(f"ValidationError: {ex.errors()[0]['msg']}")

ValidationError: Input should be 'receita', 'preco', 'substituicao' or 'fora_do_escopo'


## Fechamento do intermediário

| Notebook | O que garantiu | O que ainda falta → onde |
|---|---|---|
| 04 | dados reais via ferramentas, com guardrail na execução | dinheiro em `float`; limite de custo → **09** |
| 05 | regras de negócio + retry com feedback | regras fora do contrato, sem contexto → **07** |
| 06 | intenção → rota, ferramentas por rota, cobertura | quem pode o quê (permissões), enum aberto → **08** |

No avançado, o modelo é **simulado** (formato da API da OpenAI, sem rede) para que tudo rode e possa ser testado; uma célula no
10 mostra como trocar pelo cliente real.

➡️ [07 — contrato e contexto](../03_avancado/07_contrato_e_contexto.ipynb)